# 07.P · Mini-project: the December stock forecast

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/07-forecasting/project/project.ipynb)

Read the [README](README.md) for the brief and rubric. Fill in every `TODO`. The setup cell gives you the weekly series, the backtest helper and every model from lessons 07.2–07.5.

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

# The two sheets overlap: "Year 2009-2010" runs to 9 Dec 2010, "Year 2010-2011" starts 1 Dec 2010.
older = load_retail("Year 2009-2010")
newer = load_retail("Year 2010-2011")
raw = pd.concat([older[older["InvoiceDate"] < "2010-12-01"], newer], ignore_index=True)

def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
    return out.assign(Revenue=out["Quantity"] * out["Price"])

sales = clean_retail(raw)
sales = sales[sales["Quantity"] < 70_000]          # two one-off orders, both cancelled the same day
END = pd.Timestamp("2011-12-04")                    # last complete week (Mon 28 Nov - Sun 4 Dec)
sales = sales[sales["InvoiceDate"] < END + pd.Timedelta(days=1)]

rev = sales.set_index("InvoiceDate")["Revenue"]
daily = rev.resample("D").sum()                     # every calendar day, 0 when closed
weekly = rev.resample("W").sum()                    # weeks end on Sunday
monthly = rev.resample("ME").sum()                  # month-end dates
monthly = monthly[monthly.index <= "2011-11-30"]    # December 2011 has only 4 days: drop it
print(f"{len(sales):,} order lines, {daily.index[0].date()} to {daily.index[-1].date()}")
print(f"{len(daily)} days, {len(weekly)} weeks, {len(monthly)} months")

H = 5                                                   # forecast 5 weeks ahead
ORIGINS = pd.date_range("2011-04-03", periods=8, freq="4W-SUN")      # 3 Apr ... 16 Oct 2011

def backtest(forecast_fn, series=None, origins=ORIGINS, h=H):
    """Rolling-origin backtest: at each origin, fit on the past, forecast h weeks, score MAE."""
    series = weekly if series is None else series
    rows = []
    for origin in origins:
        train = series[series.index <= origin]
        test = series[series.index > origin].iloc[:h]
        pred = np.asarray(forecast_fn(train, len(test)))
        rows.append({"origin": origin.date(), "MAE": np.abs(test.values - pred).mean()})
    return pd.DataFrame(rows)

def naive(train, h):          return np.repeat(train.iloc[-1], h)
def mean_fc(train, h):        return np.repeat(train.mean(), h)
def seasonal_naive(train, h): return train.iloc[-52:-52 + h].values
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return train.iloc[-1] + slope * np.arange(1, h + 1)

from statsmodels.tsa.holtwinters import ExponentialSmoothing

def holt_winters(train, h):
    """Additive Holt-Winters, seeded with the first year as the starting seasonal pattern."""
    first = train.iloc[:52]
    model = ExponentialSmoothing(
        train, trend="add", damped_trend=True, seasonal="add", seasonal_periods=52,
        initialization_method="known", initial_level=first.mean(), initial_trend=0.0,
        initial_seasonal=(first - first.mean()).values).fit()
    return model.forecast(h).values

import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

def sarima(train, h):
    """SARIMA(1,0,1)(0,1,0,52) with a constant: last year's week, corrected by recent errors."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # statsmodels start-value notices, not errors
        model = SARIMAX(train, order=(1, 0, 1), seasonal_order=(0, 1, 0, 52), trend="c").fit(disp=False)
    return model.forecast(h).values

from sklearn.ensemble import HistGradientBoostingRegressor

def daily_table(d):
    """Turn the daily series into a table. Every lag is >= 35 days, so 5 weeks ahead is honest."""
    t = pd.DataFrame({"y": d})
    t["lag_35"] = d.shift(35)
    t["mean_28_lag_35"] = d.shift(35).rolling(28).mean()
    t["dow"] = d.index.day_of_week
    t["month"] = d.index.month
    t["day_of_year"] = d.index.day_of_year
    t["days_to_xmas"] = (pd.to_datetime(d.index.year.astype(str) + "-12-25") - d.index).days
    return t

TABLE = daily_table(daily)
FEATURES = [c for c in TABLE.columns if c != "y"]

def gbm(train, h):
    """Train on daily rows up to the origin, predict the next h weeks of days, sum to weeks."""
    origin = train.index[-1]
    past = TABLE[TABLE.index <= origin].dropna()
    future = TABLE[TABLE.index > origin].iloc[: 7 * h]
    model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=10, random_state=0)
    model.fit(past[FEATURES], past["y"])
    pred = pd.Series(model.predict(future[FEATURES]), index=future.index).clip(lower=0)
    pred[future["dow"] == 5] = 0                       # ShopWave never trades on Saturdays
    return pred.resample("W").sum().values[:h]

def recursive_gbm(train, h):
    """One-day-ahead model on lag_1 and lag_7; forecast day by day, feeding predictions back in."""
    origin = train.index[-1]
    d = daily[daily.index <= origin].copy()
    t = pd.DataFrame({"y": d, "lag_1": d.shift(1), "lag_7": d.shift(7),
                      "dow": d.index.day_of_week, "month": d.index.month}).dropna()
    feats = ["lag_1", "lag_7", "dow", "month"]
    model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=10, random_state=0).fit(t[feats], t["y"])
    for day in pd.date_range(origin + pd.Timedelta(days=1), periods=7 * h, freq="D"):
        row = pd.DataFrame({"lag_1": [d.iloc[-1]], "lag_7": [d.iloc[-7]],
                            "dow": [day.day_of_week], "month": [day.month]})
        d.loc[day] = 0.0 if day.day_of_week == 5 else max(model.predict(row)[0], 0.0)
    return d[d.index > origin].resample("W").sum().values[:h]

## Task 1: the split

It is Sunday 30 October 2011. Split `weekly` into `train` (up to and including 30 October) and `test` (the 5 weeks after). Do not look at `test` again until Task 6.

In [ ]:
# TODO

## Task 2: honest selection origins

The lesson backtest uses origins up to 16 October, whose forecasts run into November. Which origins can you use for model selection without touching the test weeks? Build `selection_origins`.

In [ ]:
# TODO

## Task 3: backtest the candidates

Backtest at least: seasonal naive, Holt-Winters, SARIMA, GBM direct, GBM recursive. Add one idea of your own (for example, the average of two models). Show mean and worst-origin MAE.

In [ ]:
# TODO

## Task 4: choose

Pick one model **using only the backtest**. Write two sentences on why.

In [ ]:
# TODO

## Task 5: forecast with a range

Forecast the 5 test weeks with your chosen model. Give each week a low/high range based on the model's own backtest errors (for example, ± the 80th percentile of its absolute weekly errors).

In [ ]:
# TODO

## Task 6: the reveal

Now compare with `test`. Report MAE, the 5-week total error, and how many weeks fell inside your range. How did the other candidates do?

In [ ]:
# TODO

## Task 7: the recommendation

Write a short paragraph to the buying team: how much to plan for each week, which weeks need safety stock, and how confident you are.

*Your recommendation here.*